# Урок 1.5. Установка и запуск LLM локально с помощью Ollama

**Модуль 1 «Введение в LLM» · Курс «LLM для Python-разработчиков»**

Четыре урока мы обещали: «Ollama поставим в 1.5». Этот момент настал. Сегодня вы установите локальный LLM-сервер, скачаете модель — и все «спящие» ячейки уроков 1.2 и 1.4 оживут. С этого урока весь код курса работает по-настоящему.

## 🎯 Цели урока

После этого урока вы сможете:

- объяснить **архитектуру Ollama** и почему её называют «Docker для LLM»;
- **установить** Ollama на Windows (и знать, как на macOS/Linux) и скачать модель;
- разговаривать с моделью через **CLI и оба API** (родной и OpenAI-совместимый);
- **измерять производительность**: prefill и генерация в токенах/с — и сверять с прикидками урока 1.3;
- управлять моделями и памятью: `keep_alive`, выгрузка, переменные окружения, диск;
- понимать **контекстное окно** Ollama и опасность тихой обрезки промпта.

## План

1. Что такое Ollama
2. Установка на Windows
3. Скачиваем модель
4. Первый разговор: CLI и API
5. Замеряем производительность
6. Управление моделями и памятью
7. Контекстное окно
8. Оживляем уроки 1.2 и 1.4

⏱ **Время:** ~60–90 минут (включая ~10 минут на скачивание модели).

> 💡 Этот ноутбук — интерактивный чек-лист: ячейки сами проверяют, на каком шаге установки вы находитесь, и подсказывают следующий. Выполняйте по порядку; после установки недостающего просто перезапустите ячейку проверки.

## 1. Что такое Ollama

Ollama — это локальный сервер моделей. Аналогия, которая сразу всё объясняет бэкендеру — **Docker, только для LLM**:

| Docker | Ollama | смысл |
|---|---|---|
| `docker pull nginx` | `ollama pull qwen2.5:3b` | скачать образ/модель из реестра |
| `docker images` | `ollama list` | что скачано на диск |
| `docker ps` | `ollama ps` | что сейчас загружено в память |
| `docker run -it` | `ollama run qwen2.5:3b` | интерактивный запуск |
| `docker stop` | `ollama stop qwen2.5:3b` | выгрузить из памяти |
| `docker rmi` | `ollama rm qwen2.5:3b` | удалить с диска |
| `docker inspect` | `ollama show qwen2.5:3b` | детали образа/модели |

Даже устройство похоже: модели хранятся **слоями** (веса, шаблон промпта, параметры — отдельные блобы), у моделей есть **теги**, а скачивание докачивается после обрыва.

Архитектура:

```text
  ваш код / CLI / Jupyter
          |
          | HTTP: localhost:11434
          v
+---------------------------+
|  сервер Ollama            |   процесс в трее Windows
|  - REST API (/api, /v1)   |
|  - менеджер моделей       |
|  - планировщик памяти     |
+---------------------------+
          |
          v
   движок llama.cpp          выполняет модель на CPU/GPU
          |
          v
   ~/.ollama/models          скачанные веса на диске
```

Важное следствие: **сервер и модели — разные вещи.** Сервер может работать, пока не скачано ни одной модели, и наоборот — модели лежат на диске, а сервер выключен. Проверять надо и то и другое, чем мы и займёмся.

## 2. Установка на Windows

**Вариант А — установщик (проще всего):**

1. Откройте [ollama.com/download](https://ollama.com/download) → **Download for Windows**.
2. Запустите `OllamaSetup.exe` и дождитесь окончания (прав администратора не требуется).

**Вариант Б — через winget:**

```powershell
winget install Ollama.Ollama
```

Требования: Windows 10+, пара гигабайт на приложение плюс место под модели. **GPU не нужен** — на CPU всё работает, просто медленнее (мы это уже посчитали в уроке 1.3).

После установки Ollama **запускается сама** и добавляется в автозагрузку — в трее появится иконка ламы. Сервер слушает `http://localhost:11434`.

**Проверка** — в *новом* окне терминала:

```powershell
ollama --version
```

И в браузере откройте `http://localhost:11434` — увидите «Ollama is running».

На macOS: `brew install ollama`, на Linux: `curl -fsSL https://ollama.com/install.sh | sh`.

Теперь пусть ноутбук сам проверит, что у вас получилось:

In [ ]:
# Зависимости урока (обе уже ставились в прошлых уроках)
# %pip install -q httpx openai

In [ ]:
import shutil
import subprocess

OLLAMA_CLI = shutil.which("ollama")

if OLLAMA_CLI:
    version = subprocess.run(["ollama", "--version"],
                             capture_output=True, text=True).stdout.strip()
    print("Шаг 1/3 пройден. CLI найден:", OLLAMA_CLI)
    print(version)
else:
    print("CLI не найден.")
    print("1) Установите Ollama (инструкция выше).")
    print("2) Перезапустите ядро Jupyter: свежеустановленный PATH подхватывается")
    print("   только при старте процесса (частые грабли!).")
    print("3) Выполните эту ячейку снова.")

In [ ]:
import httpx

OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen2.5:3b"

try:
    r = httpx.get(OLLAMA_URL, timeout=2)
    SERVER_UP = True
    print("Шаг 2/3 пройден. Сервер отвечает:", r.text)
except (httpx.ConnectError, httpx.TimeoutException):
    SERVER_UP = False
    print("Сервер не отвечает. Если Ollama установлена, запустите её:")
    print("  - найдите Ollama в меню Пуск (в трее появится иконка ламы), или")
    print("  - выполните в терминале: ollama serve")
    print("Затем перезапустите эту ячейку.")

## 3. Скачиваем модель

Наш рабочий инструмент на весь курс — **`qwen2.5:3b`**: 3 миллиарда параметров, ~1.9 ГБ на диске, прилично говорит по-русски и помещается практически в любую машину (проверьте себя ячейкой из урока 1.3). Скачивание — в терминале:

```powershell
ollama pull qwen2.5:3b
```

Про **имена моделей**: полный формат тега — `семейство:размер-вариант-квантизация`, например `qwen2.5:3b-instruct-q4_K_M`. Короткий тег `qwen2.5:3b` — это псевдоним такого длинного: instruct-вариант (про base/instruct — урок 1.6) в квантизации Q4 (урок 1.3). Дефолты подобраны разумно, поэтому короткого тега почти всегда достаточно.

Модели на вырост — по вашей RAM-математике из урока 1.3:

| модель | на диске | в RAM | зачем |
|---|---|---|---|
| `llama3.2:1b` | ~1.3 ГБ | ~2 ГБ | совсем слабое железо, быстрые эксперименты |
| **`qwen2.5:3b`** | ~1.9 ГБ | ~3–4 ГБ | основная модель курса |
| `qwen2.5:7b` | ~4.7 ГБ | ~6–8 ГБ | качественнее, ощутимо медленнее на CPU |
| `qwen2.5:14b` | ~9 ГБ | ~11+ ГБ | для 16+ ГБ RAM; на CPU уже задумчива |

Реестр моделей — [ollama.com/library](https://ollama.com/library).

In [ ]:
MODEL_READY = False

if SERVER_UP:
    tags = httpx.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()
    installed = [m["name"] for m in tags.get("models") or []]
    print("Скачанные модели:", ", ".join(installed) if installed else "пока нет")
    MODEL_READY = MODEL in installed
    if MODEL_READY:
        print(f"Шаг 3/3 пройден. {MODEL} на месте — поехали!")
    else:
        print(f"\nМодель {MODEL} не найдена. Скачайте её в терминале:")
        print(f"  ollama pull {MODEL}")
        print("(~1.9 ГБ. Можно и прямо из ноутбука - раскомментируйте строку ниже.)")
        # !ollama pull qwen2.5:3b
else:
    print("[сервер не запущен — вернитесь к предыдущей ячейке]")

## 4. Первый разговор: CLI и API

Самый быстрый способ пообщаться с моделью — интерактивный чат в терминале:

```powershell
ollama run qwen2.5:3b
```

Пишете сообщение — получаете ответ (стримингом, как в уроке 1.4). Команда `/bye` — выход, `/?` — список команд. CLI удобен для быстрой проверки «жива ли модель и насколько она вменяема», но приложения разговаривают через API — так что вернёмся в Python:

In [ ]:
if MODEL_READY:
    r = httpx.post(f"{OLLAMA_URL}/api/chat", json={
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "Ты — лаконичный дружелюбный ассистент."},
            {"role": "user", "content": "Поздравь меня: я только что запустил LLM на собственном компьютере!"},
        ],
        "options": {"temperature": 0.8, "num_predict": 100},
        "stream": False,
    }, timeout=300)
    r.raise_for_status()
    print(r.json()["message"]["content"])
else:
    print("[пропущено: модель ещё не готова — пройдите шаги выше]")

> **Пример вывода:**
>
> «Поздравляю! Теперь у вас есть собственная языковая модель — прямо на вашем компьютере, без облака, подписок и передачи данных наружу. Отличный шаг!»

Если первый запрос занял заметно дольше последующих — это нормально: Ollama **загружала модель с диска в память**. Через несколько минут простоя она выгрузится, и первый запрос снова будет «холодным» — разберём это в разделе про память.

Напомню из урока 1.4: у Ollama **два API**. Родной (`/api/chat`) — все настройки: `num_ctx`, `keep_alive`, метрики производительности в ответе. OpenAI-совместимый (`/v1/chat/completions`) — переносимость: код не отличается от кода для облака. Проверим и его — тем самым `LLMClient`-паттерном из урока 1.4:

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=f"{OLLAMA_URL}/v1", api_key="ollama", timeout=120.0)

if MODEL_READY:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Одним предложением: кто ты и на чём работаешь?"}],
        max_tokens=60,
    )
    print(resp.choices[0].message.content)
    print(f"usage: {resp.usage.prompt_tokens} in / {resp.usage.completion_tokens} out")
else:
    print("[пропущено: модель ещё не готова]")

## 5. Замеряем производительность

В уроке 1.3 мы *прикидывали* скорость на бумаге. Теперь можно **измерить**: родной API возвращает с каждым ответом телеметрию (всё в наносекундах):

| поле | что это |
|---|---|
| `load_duration` | загрузка модели в память (0, если уже была загружена) |
| `prompt_eval_count` / `prompt_eval_duration` | токенов промпта обработано / за сколько (**prefill**) |
| `eval_count` / `eval_duration` | токенов ответа сгенерировано / за сколько (**decode**) |
| `total_duration` | всё вместе |

Скорость = количество / длительность × 10⁹. Напишем бенчмарк:

In [ ]:
def bench(prompt: str, num_predict: int = 200) -> dict | None:
    """Замер prefill и генерации через телеметрию родного API."""
    if not MODEL_READY:
        print("[пропущено: модель ещё не готова]")
        return None
    r = httpx.post(f"{OLLAMA_URL}/api/chat", json={
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "options": {"num_predict": num_predict, "temperature": 0.7},
        "stream": False,
    }, timeout=600)
    r.raise_for_status()
    d = r.json()
    prefill_s = d["prompt_eval_duration"] / 1e9
    decode_s = d["eval_duration"] / 1e9
    return {
        "промпт, токенов":    d["prompt_eval_count"],
        "prefill, ток/с":     round(d["prompt_eval_count"] / max(prefill_s, 1e-9)),
        "ответ, токенов":     d["eval_count"],
        "генерация, ток/с":   round(d["eval_count"] / max(decode_s, 1e-9)),
        "загрузка модели, с": round(d["load_duration"] / 1e9, 2),
        "всего, с":           round(d["total_duration"] / 1e9, 1),
    }

report = bench("Расскажи подробно, что такое кэширование в веб-приложениях.")
if report:
    for key, value in report.items():
        print(f"{key:<20} {value}")

> **Пример вывода** (десктопный CPU, без GPU):
>
> ```text
> промпт, токенов      31
> prefill, ток/с       142
> ответ, токенов       200
> генерация, ток/с     22
> загрузка модели, с   0.08
> всего, с             9.4
> ```

Типичные значения для 3B Q4 на современном CPU: prefill 50–200 ток/с, генерация 10–30 ток/с. Сравните со своими прикидками из урока 1.3 — и с ожиданием времени ответа: 300-токенный ответ при 20 ток/с — это 15 секунд, вот вам и мотивация для стриминга из урока 1.4.

> ⚠️ **Кэш префикса.** Если прогнать `bench` дважды с одним и тем же промптом, второй раз покажет фантастический prefill (а `prompt_eval_count` станет крошечным): Ollama кэширует уже обработанное начало диалога. Для честных замеров меняйте текст промпта. Для приложений же это подарок — длинный системный промпт обрабатывается только один раз.

## 6. Управление моделями и памятью

Жизненный цикл модели в памяти:

```text
запрос --> [модель на диске] --load: секунды--> [модель в RAM] --> ответы: быстро
                                                      |
                                        нет запросов keep_alive (5 минут)
                                                      v
                                              выгрузка из RAM
```

Это разумный дефолт для ноутбука (модель не держит гигабайты RAM зря), но в проде «холодный старт» после каждого затишья — неприятный сюрприз для первого пользователя. Управляется параметром `keep_alive` в запросе (`"keep_alive": "30m"`, `0` — выгрузить сразу, `-1` — держать вечно) или глобально переменной окружения.

Переменные окружения сервера (задаются в системных настройках Windows, после чего Ollama надо перезапустить из трея):

| переменная | что делает | дефолт |
|---|---|---|
| `OLLAMA_HOST` | адрес/порт сервера | `127.0.0.1:11434` |
| `OLLAMA_MODELS` | папка моделей на диске | `%USERPROFILE%\.ollama\models` |
| `OLLAMA_KEEP_ALIVE` | сколько держать модель в RAM | `5m` |
| `OLLAMA_CONTEXT_LENGTH` | рабочий контекст по умолчанию | 4096 (в свежих версиях) |
| `OLLAMA_NUM_PARALLEL` | параллельные запросы к одной модели | 1/авто |

`OLLAMA_MODELS` особенно важна, если системный диск маленький: пара скачанных 7B–14B моделей — это десятки гигабайт.

Что сейчас в памяти — покажет `ollama ps` или его API-аналог:

In [ ]:
if SERVER_UP:
    ps = httpx.get(f"{OLLAMA_URL}/api/ps", timeout=5).json()
    loaded = ps.get("models") or []
    if not loaded:
        print("Сейчас в памяти нет ни одной модели (все выгружены).")
    for m in loaded:
        print(f"{m['name']}: {m['size'] / 2**30:.1f} ГБ в RAM, "
              f"выгрузится в {m['expires_at'][:19]}")
else:
    print("[сервер не запущен]")

## 7. Контекстное окно: тихая ловушка

Из урока 1.1 вы знаете про контекстное окно модели. У Ollama есть нюанс, который ловит почти всех: **рабочий контекст задаёт сервер, а не модель**. Модель может поддерживать 32k токенов, но Ollama по умолчанию выделяет под контекст меньше (обычно 4096) — потому что KV-кэш (урок 1.3) ест RAM пропорционально длине контекста.

Самое коварное: промпт длиннее рабочего контекста **не вызывает ошибку** — он тихо обрезается, и модель просто не видит часть текста. Для чата это лёгкая деградация, а для RAG (скоро!) — катастрофа: вы кладёте в промпт документы, а модель получает половину. Защита: следить за `prompt_eval_count` и явно задавать `options.num_ctx` в родном API.

Посмотрим, что заявляет наша модель и её конфигурация:

In [ ]:
if MODEL_READY:
    info = httpx.post(f"{OLLAMA_URL}/api/show", json={"model": MODEL}, timeout=10).json()
    details = info.get("details", {})
    print("Параметров:  ", details.get("parameter_size"))
    print("Квантизация: ", details.get("quantization_level"))

    ctx = next((v for k, v in (info.get("model_info") or {}).items()
                if k.endswith(".context_length")), "?")
    print("Максимум контекста у модели:", ctx, "токенов")

    print("\nРабочий контекст сервера — другое число! Задать своё на один запрос:")
    print('  "options": {"num_ctx": 8192}   # в родном API; помните про RAM')
else:
    print("[пропущено: модель ещё не готова]")

## 8. Оживляем уроки 1.2 и 1.4 ✨

Ради этого момента всё и затевалось. Теперь, когда модель работает, вернитесь и перезапустите целиком (Kernel → Restart & Run All):

- **Урок 1.2** — там ждут живой чат, эксперименты с `temperature` (запустите ячейку с temperature=1.5 несколько раз!), JSON-режим с Pydantic и — самое интересное — **красная команда против бота «Ромашки»**: инъекции теперь будут атаковать настоящую модель.
- **Урок 1.4** — первый запрос через SDK, **стриминг** (посмотрите, как текст печатается токен за токеном), async-классификация отзывов и упражнение с TTFT-секундомером.
- В уроке 1.1 тоже была опциональная ячейка-«превью» с Ollama — теперь и она работает.

Это отличная тренировка: перечитать код недельной давности и увидеть, как он оживает, — лучший способ закрепить материал.

## ⚠️ Частые ошибки

1. **Начать с гигантской модели** («скачаю-ка 70B, чего мелочиться»): 40 ГБ трафика — и она не влезет в RAM. Сначала RAM-математика из урока 1.3, потом `pull`.
2. **«Connection refused» = паника.** Чаще всего сервер просто не запущен: посмотрите на трей, запустите Ollama из меню Пуск или `ollama serve`. Сервер и модели — разные вещи.
3. **Jupyter не видит свежеустановленный `ollama`.** PATH подхватывается при старте процесса — перезапустите ядро/терминал.
4. **Тихая обрезка промпта.** Текст длиннее рабочего контекста молча теряется — ни ошибки, ни предупреждения. Контролируйте `prompt_eval_count`, задавайте `num_ctx`.
5. **Бенчмарк одним и тем же промптом.** Кэш префикса даст сказочный prefill. Для замеров меняйте текст, для приложений — радуйтесь кэшу.
6. **Пугаться медленного первого запроса после простоя.** Это `load_duration` (холодный старт по `keep_alive`), а не деградация сервиса.
7. **Забыть про диск.** Модели живут в `~/.ollama/models` на системном диске; пара экспериментов с 7B–14B — и место кончилось. `OLLAMA_MODELS` переносит хранилище.

## 🏋️ Упражнения

### Упражнение 1 — бенчмарк вашей машины 📊

Прогоните `bench` на промптах трёх разных длин (короткий вопрос, ~500 токенов, ~2000 токенов) и сведите в таблицу: токены промпта, prefill ток/с, генерация ток/с. Промпты должны *отличаться с первых слов* (помните про кэш префикса). Совпали ли числа с прикидками урока 1.3? Что деградирует с ростом промпта?

### Упражнение 2 — холодный и тёплый старт 🥶

Измерьте разницу: выгрузите модель (`keep_alive: 0` в запросе или `ollama stop`), сделайте «холодный» запрос и сразу — «тёплый». Сравните `load_duration` и `total_duration`. Сформулируйте вывод для продакшена.

### Упражнение 3 — healthcheck для будущих сервисов 🩺

Напишите функцию `ollama_healthcheck(base_url, model) -> dict` с тремя уровнями проверки: сервер отвечает → модель скачана → генерация работает (крошечный запрос с `num_predict: 3`). Верните словарь со статусами и деталями. Эта функция станет эндпоинтом `/healthz`, когда мы завернём LLM в FastAPI-сервис.

## 📝 Мини-квиз

**1. Что общего у `ollama pull` и `docker pull`?**

<details><summary>Ответ</summary>

Оба качают артефакт из реестра по имени с тегом, хранят его локально слоями, докачивают после обрыва и переиспользуют уже скачанные слои. Модель Ollama — это «образ»: веса + шаблон промпта + параметры.

</details>

**2. Сервер отвечает «Ollama is running», но `/api/tags` возвращает пустой список. Что это значит?**

<details><summary>Ответ</summary>

Сервер работает, но ни одна модель не скачана — это независимые вещи. Лечится `ollama pull <модель>`.

</details>

**3. Почему первый запрос после долгого перерыва заметно медленнее остальных?**

<details><summary>Ответ</summary>

По истечении `keep_alive` (по умолчанию 5 минут без запросов) модель выгружается из RAM. Следующий запрос платит за загрузку с диска — это видно в поле `load_duration`.

</details>

**4. Как из ответа родного API посчитать скорость генерации?**

<details><summary>Ответ</summary>

`eval_count / eval_duration × 10⁹` — токенов в секунду (длительности в ответе даны в наносекундах). Аналогично prefill: `prompt_eval_count / prompt_eval_duration × 10⁹`.

</details>

**5. Что произойдёт с промптом длиннее рабочего контекста?**

<details><summary>Ответ</summary>

Он **тихо обрежется** — без ошибки и предупреждения; модель просто не увидит часть текста. Поэтому за `prompt_eval_count` нужно следить, а для длинных промптов — явно поднимать `num_ctx` (помня, что KV-кэш ест RAM).

</details>

## ✅ Решения упражнений

### Решение 1 — бенчмарк трёх длин

In [ ]:
if MODEL_READY:
    # у каждого промпта своё первое слово - кэш префикса не сработает
    fact_a = "Кэш хранит горячие данные в памяти и снижает нагрузку на базу. "
    fact_b = "Брокер сообщений развязывает сервисы и сглаживает пики нагрузки. "
    prompts = {
        "короткий":  "Что такое очередь сообщений?",
        "~500 ток":  "Первый текст. " + fact_a * 18 + "Суммируй сказанное одним абзацем.",
        "~2000 ток": "Второй текст. " + fact_b * 90 + "Суммируй сказанное одним абзацем.",
    }
    print(f"{'промпт':<11} {'токенов':>8} {'prefill ток/с':>14} {'ген. ток/с':>11}")
    print("-" * 48)
    for label, prompt in prompts.items():
        r = bench(prompt, num_predict=80)
        print(f"{label:<11} {r['промпт, токенов']:>8} "
              f"{r['prefill, ток/с']:>14} {r['генерация, ток/с']:>11}")
else:
    print("[пропущено: модель ещё не готова]")

Что обычно видно в результатах: скорость prefill (ток/с) с ростом промпта держится или даже растёт (лучше утилизируется батчинг), но *абсолютное* время prefill растёт линейно — 2000 токенов на CPU это уже десятки секунд ожидания до первого токена. Генерация от длины промпта почти не зависит. И проверьте: если у «~2000 ток» `prompt_eval_count` подозрительно меньше ожидаемого — вы упёрлись в рабочий контекст (раздел 7).

### Решение 2 — холодный и тёплый старт

In [ ]:
if MODEL_READY:
    def timed_request(question: str, keep_alive=None) -> tuple[float, float]:
        payload = {
            "model": MODEL,
            "messages": [{"role": "user", "content": question}],
            "options": {"num_predict": 30},
            "stream": False,
        }
        if keep_alive is not None:
            payload["keep_alive"] = keep_alive
        d = httpx.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=600).json()
        return d["load_duration"] / 1e9, d["total_duration"] / 1e9

    # шаг 1: просим выгрузить модель сразу после ответа
    timed_request("Скажи «ок».", keep_alive=0)

    # шаг 2: холодный запрос - модель загружается с диска
    cold_load, cold_total = timed_request("Что такое индекс в базе данных?")

    # шаг 3: тёплый запрос - модель уже в памяти
    warm_load, warm_total = timed_request("Что такое транзакция?")

    print(f"холодный: загрузка {cold_load:5.2f} с, всего {cold_total:5.2f} с")
    print(f"тёплый:   загрузка {warm_load:5.2f} с, всего {warm_total:5.2f} с")
else:
    print("[пропущено: модель ещё не готова]")

Вывод для продакшена: холодный старт добавляет секунды к времени ответа, и платит за них самый невезучий пользователь. Если сервис живёт на локальной модели — держите её прогретой (`OLLAMA_KEEP_ALIVE=-1` или периодический «пинг»), а `/healthz` должен делать настоящий крошечный запрос, а не только проверять порт — иначе мониторинг будет зелёным, а пользователи будут ждать.

### Решение 3 — healthcheck

In [ ]:
def ollama_healthcheck(base_url: str = "http://localhost:11434",
                       model: str = "qwen2.5:3b",
                       timeout: float = 60.0) -> dict:
    """Трёхуровневая проверка: сервер -> модель скачана -> генерация работает."""
    status = {"server": False, "model": False, "generation": False, "detail": "ok"}

    try:
        httpx.get(base_url, timeout=5)
        status["server"] = True
    except (httpx.ConnectError, httpx.TimeoutException) as e:
        status["detail"] = f"сервер недоступен: {type(e).__name__}"
        return status

    try:
        tags = httpx.get(f"{base_url}/api/tags", timeout=5).json()
        names = [m["name"] for m in tags.get("models") or []]
        status["model"] = model in names
        if not status["model"]:
            status["detail"] = f"модель {model} не скачана (есть: {names or 'ничего'})"
            return status

        r = httpx.post(f"{base_url}/api/chat", json={
            "model": model,
            "messages": [{"role": "user", "content": "ping"}],
            "options": {"num_predict": 3},
            "stream": False,
        }, timeout=timeout)
        r.raise_for_status()
        status["generation"] = True
    except Exception as e:  # healthcheck-у можно ловить широко: его дело - доложить
        status["detail"] = f"{type(e).__name__}: {e}"

    return status

result = ollama_healthcheck()
healthy = result["server"] and result["model"] and result["generation"]
print("СТАТУС:", "здоров" if healthy else "нездоров")
for key, value in result.items():
    print(f"  {key}: {value}")

Заметьте: эта ячейка работает при **любом** состоянии системы — она и написана для того, чтобы честно доложить, что не так. Три уровня дают три разных диагноза для дежурного: «подними сервер», «скачай модель», «модель есть, но генерация падает — смотри логи». Когда в финальном проекте мы завернём всё в FastAPI, эта функция почти без изменений станет эндпоинтом `/healthz`.

## 🎓 Итоги

- Ollama — «Docker для LLM»: реестр, теги, слои, `pull/list/ps/run/stop/rm`. Сервер на `localhost:11434` и скачанные модели — независимые вещи, проверять надо обе.
- Наша модель курса — `qwen2.5:3b` (~1.9 ГБ, Q4, instruct). Короткий тег скрывает `‑instruct‑q4_K_M`.
- Родной API отдаёт **телеметрию**: prefill и генерация в ток/с считаются из `*_count / *_duration`. На CPU: prefill 50–200, генерация 10–30 ток/с для 3B — теперь у вас есть свои цифры вместо прикидок.
- Память: модель выгружается по `keep_alive` (5 мин) — отсюда холодные старты; рабочий контекст задаёт сервер, и слишком длинный промпт **тихо обрезается**.
- Уроки 1.2 и 1.4 теперь полностью живые — перезапустите их.

**Дальше — урок 1.6 «Base-модели и instruct-модели»:** разберёмся, что означает суффикс `-instruct`, который мы всё это время молча скачивали, и почему «сырая» base-модель ведёт себя совсем не как чат-бот.

## 📚 Что почитать

- [Документация API Ollama](https://github.com/ollama/ollama/blob/main/docs/api.md) — все эндпоинты и поля телеметрии
- [FAQ Ollama](https://github.com/ollama/ollama/blob/main/docs/faq.md) — keep_alive, переменные окружения, GPU
- [Ollama на Windows](https://github.com/ollama/ollama/blob/main/docs/windows.md) — особенности платформы
- [Библиотека моделей](https://ollama.com/library) — реестр с тегами и размерами
- [OpenAI compatibility](https://github.com/ollama/ollama/blob/main/docs/openai.md) — что поддержано в `/v1`